In [ ]:
%load_ext sql
%sql duckdb:////workspace/ICOCO.duckdb

In [ ]:
# Close the connection FIRST to flush the WAL file (ICOCO.duckdb.wal)
# and avoid data corruption from an aborted transaction.
# NOTE: the connection name (URL) is required as an argument.
%sql --close duckdb:////workspace/ICOCO.duckdb

In [ ]:
# Reconnect with a fresh connection (recovers/checkpoints the WAL)
%sql duckdb:////workspace/ICOCO.duckdb

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS bronze;

# 1. Add Zenodo dataset into DuckDB

In [ ]:
import csv
import re

# DuckDB 1.1.3's read_csv has NO 'encoding' parameter (unlike older versions),
# so the cp1252 file must be converted to UTF-8 first. The raw file also contains
# control characters and unescaped double-quotes that break DuckDB's strict CSV
# parser, so we re-emit a properly-quoted UTF-8 CSV with Python's csv module.

SRC = '/workspace/data/RecipeZenodo.csv'
TMP = '/workspace/data/RecipeZenodo_utf8.csv'
OUT = '/workspace/data/RecipeZenodo_final.csv'

# Step 1: cp1252 -> UTF-8 and strip control characters (e.g. NUL)
CTRL = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')
with open(SRC, 'rb') as fh, open(TMP, 'w', newline='', encoding='utf-8') as fo:
    for line in fh:
        fo.write(CTRL.sub('', line.decode('cp1252')))

# Step 2: re-quote fields properly (fixes unescaped double-quotes)
with open(TMP, newline='', encoding='utf-8') as fh, open(OUT, 'w', newline='', encoding='utf-8') as fo:
    reader = csv.reader(fh, delimiter=';', quotechar='"')
    writer = csv.writer(fo, delimiter=';', quotechar='"', quoting=csv.QUOTE_MINIMAL, lineterminator='\n')
    for row in reader:
        if len(row) != 18:  # skip malformed rows
            continue
        writer.writerow(row)

print('Preprocessed CSV ready:', OUT)

In [ ]:
%%sql
CREATE OR REPLACE TABLE bronze.RecipeZenodo AS
SELECT * FROM read_csv(
    '/workspace/data/RecipeZenodo_final.csv',
    delim=';',
    header=true,
    quote='"',
    ignore_errors=true,
    null_padding=true,
    parallel=false
);

In [ ]:
%%sql
SELECT COUNT(*) AS row_count FROM bronze.RecipeZenodo;